<a href="https://colab.research.google.com/github/samuelnassam/Facteur-Momentum-et-Rendements-Boursiers-Futurs/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf

data est un DataFrame Pandas : chaque colonne est une action, chaque ligne est une date, chaque valeur est le prix de clôture ce jour-là. C'est la structure de base sur laquelle tout le projet va reposer.

Ici : ["MC.PA", "OR.PA"] correspond à LVMH et L'Oréal (chaque action a un code appelé "ticker" sur les marchés). start et end définissent la période. Cette ligne télécharge les prix de clôture quotidiens de ces deux actions sur 5 ans et les range dans une variable data.

In [ ]:
data = yf.download(["MC.PA", "OR.PA"], start="2019-01-01", end="2024-01-01")["Close"]

/tmp/ipykernel_721/3757361484.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(["MC.PA", "OR.PA"], start="2019-01-01", end="2024-01-01")["Close"]
[*********************100%***********************]  2 of 2 completed


In [ ]:
print(data.head())

Ticker           MC.PA       OR.PA
Date                              
2019-01-02  222.208710  176.734451
2019-01-03  213.743195  174.609360
2019-01-04  220.322601  177.088577
2019-01-07  219.927841  174.830734
2019-01-08  226.814255  175.893250


shape  donne un couple de nombres : (nombre de lignes, nombre de colonnes) et donne le nombre de jours de cotation et d'actions.

In [ ]:
print(data.shape)

(1283, 20)


.isna() repère les valeurs manquantes (les NaN) dans le tableau, et .sum() les compte par colonne. On vérifie qu'il n'y a pas de trou dans les données avant de calculer quoi que ce soit dessus.

In [ ]:
print(data.isna().sum())

Ticker
MC.PA    0
OR.PA    0
dtype: int64


Zéro valeur manquante sur les deux colonnes — données propres, on peut avancer sans nettoyage supplémentaire pour l'instant.

In [ ]:
momentum_6m = data.pct_change(periods=126) * 100
print(momentum_6m.tail())

Ticker          MC.PA     OR.PA
Date                           
2023-12-21 -11.816642  8.130280
2023-12-22 -13.035217  6.635685
2023-12-27 -14.110005  5.057374
2023-12-28 -13.814982  6.551771
2023-12-29 -13.690184  6.285377


Ici : LVMH, L'Oréal, TotalEnergies, Air Liquide, Sanofi et BNP Paribas — un petit panier diversifié (luxe, énergie, industrie, santé, banque). On refait les deux vérifications qu'on a déjà faites (dimensions + valeurs manquantes) avant d'aller plus loin, par réflexe méthodologique.

In [ ]:
tickers = ["MC.PA", "OR.PA", "TTE.PA", "AI.PA", "SAN.PA", "BNP.PA"]
data = yf.download(tickers, start="2019-01-01", end="2024-01-01")["Close"]

print(data.shape)
print(data.isna().sum())

/tmp/ipykernel_721/2287083234.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2019-01-01", end="2024-01-01")["Close"]
[*********************100%***********************]  6 of 6 completed

(1283, 6)
Ticker
AI.PA     0
BNP.PA    0
MC.PA     0
OR.PA     0
SAN.PA    0
TTE.PA    0
dtype: int64


In [ ]:
momentum_6m = data.pct_change(periods=126) * 100
print(momentum_6m.tail())

Ticker         AI.PA     BNP.PA      MC.PA     OR.PA    SAN.PA     TTE.PA
Date                                                                     
2023-12-21  8.763084  12.567139 -11.816642  8.130280 -9.995950  20.600591
2023-12-22  8.676455  10.952627 -13.035217  6.635685 -8.631623  21.394193
2023-12-27  7.685743   9.130287 -14.110005  5.057374 -9.195521  20.289275
2023-12-28  7.164575   6.880024 -13.814982  6.551771 -9.159371  16.148446
2023-12-29  7.850576   8.118842 -13.690184  6.285377 -8.155125  16.225375


Jusqu'ici, momentum_6m regarde en arrière (le rendement des 6 derniers mois). Pour tester si le momentum a un pouvoir prédictif, il faut maintenant regarder en avant : quel a été le rendement des 6 mois suivants, à partir de chaque date. C'est cette deuxième variable qu'on va comparer au momentum.

In [ ]:
future_return_6m = (data.shift(-126) / data - 1) * 100
print(future_return_6m.tail(10))

Ticker      AI.PA  BNP.PA  MC.PA  OR.PA  SAN.PA  TTE.PA
Date                                                   
2023-12-14    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-15    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-18    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-19    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-20    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-21    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-22    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-27    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-28    NaN     NaN    NaN    NaN     NaN     NaN
2023-12-29    NaN     NaN    NaN    NaN     NaN     NaN


Pour les dates trop proches de la fin de ta période (fin décembre 2023), il n'existe pas encore de prix "126 jours après" dans les données. Pandas va donc afficher des NaN (valeurs manquantes) sur ces dernières lignes.

In [ ]:
import pandas as pd

dataset = pd.DataFrame({
    "momentum_6m": momentum_6m.stack(),
    "future_return_6m": future_return_6m.stack()
}).dropna()

print(dataset.shape)
print(dataset.head())

(6186, 2)
                   momentum_6m  future_return_6m
Date       Ticker                               
2019-07-02 AI.PA     18.541118         12.728786
           BNP.PA    14.951822         26.549602
           MC.PA     50.750982         10.307867
           OR.PA     29.297147          4.133849
           SAN.PA     6.550254         17.563904


.corr() calcule le coefficient de corrélation de Pearson entre les deux colonnes — un nombre entre -1 et +1. Proche de 0 = pas de lien linéaire ; proche de +1 = les actions à fort momentum passé ont tendance à avoir un fort rendement futur ; proche de -1 = l'inverse (effet de retour à la moyenne).

In [ ]:
correlation = dataset["momentum_6m"].corr(dataset["future_return_6m"])
print(correlation)

-0.18308652162794542


À chaque date, on classe les 6 actions selon leur momentum, on regarde qui est dans le tiers "fort momentum" et qui est dans le tiers "faible momentum", et on compare leurs rendements futurs moyens respectifs.

In [ ]:
dataset["tertile"] = dataset.groupby("Date")["momentum_6m"].transform(
    lambda x: pd.qcut(x, 3, labels=["faible", "moyen", "fort"], duplicates="drop")
)

resultats = dataset.groupby("tertile")["future_return_6m"].mean()
print(resultats)

tertile
faible    9.463763
moyen     8.619883
fort      5.183442
Name: future_return_6m, dtype: float64


/tmp/ipykernel_721/3284045810.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resultats = dataset.groupby("tertile")["future_return_6m"].mean()


groupby("Date") regroupe les lignes par date (donc les 6 actions de chaque date ensemble).
.transform(lambda x: pd.qcut(...)) découpe, pour chaque date séparément, le momentum des 6 actions en 3 groupes de taille égale (2 actions par groupe environ).
La deuxième ligne calcule le rendement futur moyen pour chaque tertile.

In [ ]:
from scipy import stats

groupe_faible = dataset[dataset["tertile"] == "faible"]["future_return_6m"]
groupe_fort = dataset[dataset["tertile"] == "fort"]["future_return_6m"]

t_stat, p_value = stats.ttest_ind(groupe_faible, groupe_fort)
print("t-statistique :", t_stat)
print("p-value :", p_value)

t-statistique : 8.203895931084197
p-value : 3.078638869181231e-16


In [ ]:
dataset = dataset.reset_index()
print(dataset["Date"].min(), dataset["Date"].max())

2019-07-02 00:00:00 2023-07-04 00:00:00


Ici, on filtre simplement le DataFrame selon la date : tout ce qui est avant le 1ᵉʳ juillet 2022 sert à entraîner le modèle, tout ce qui est après sert uniquement à l'évaluer.

In [ ]:
train = dataset[dataset["Date"] < "2022-07-01"]
test = dataset[dataset["Date"] >= "2022-07-01"]

print("Train :", train.shape)
print("Test :", test.shape)

Train : (4632, 5)
Test : (1554, 5)


Bon ratio : ~75 % en train (4632), ~25 % en test (1554) — cohérent avec ce qu'on visait.

In [ ]:
from sklearn.linear_model import LinearRegression

X_train = train[["momentum_6m"]]
y_train = train["future_return_6m"]

X_test = test[["momentum_6m"]]
y_test = test["future_return_6m"]

modele = LinearRegression()
modele.fit(X_train, y_train)

print("Coefficient :", modele.coef_)
print("Ordonnée à l'origine :", modele.intercept_)

Coefficient : [-0.09299166]
Ordonnée à l'origine : 7.534450769102488


-X_train/X_test : la variable explicative (le momentum), sous forme de tableau à une colonne — LinearRegression exige ce format, d'où les doubles crochets [["momentum_6m"]].
-y_train/y_test : la variable à prédire (le rendement futur).
modele.fit(X_train, y_train) : c'est l'entraînement — le modèle cherche la droite qui minimise l'erreur sur les données d'entraînement (exactement la descente de gradient qu'on avait vue en théorie à l'étape 5, ici résolue directement par scikit-learn).
-Le coefficient dit: pour chaque point de momentum en plus, de combien varie le rendement futur prédit.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

predictions = modele.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MAE :", mae)
print("R² :", r2)

MAE : 10.281971803425368
R² : 0.01928454358814924


Ici, predictions_baseline répète simplement la moyenne du train pour chaque observation du test.

In [ ]:
moyenne_train = y_train.mean()
predictions_baseline = [moyenne_train] * len(y_test)

mae_baseline = mean_absolute_error(y_test, predictions_baseline)

print("MAE baseline (moyenne naïve) :", mae_baseline)
print("MAE modèle (momentum)        :", mae)

MAE baseline (moyenne naïve) : 10.950779196827353
MAE modèle (momentum)        : 10.281971803425368


Le modèle fait mieux que la baseline naïve : 10,28 contre 10,95, soit une amélioration d'environ 0,67 point de pourcentage (~6 % de réduction de l'erreur). C'est modeste en valeur absolue, mais c'est exactement la bonne nouvelle qu'il fallait : le momentum apporte un vrai signal, même petit, plutôt que de ne servir à rien.

Avec 6 actions, chaque tertile ne contient que 2 titres — trop peu pour être vraiment robuste. On va élargir à une vingtaine d'actions du CAC 40, ce qui rendra les tertiles (7 actions par groupe environ) et le test statistique bien plus solides.

In [ ]:
tickers_larges = [
    "MC.PA", "OR.PA", "TTE.PA", "AI.PA", "SAN.PA", "BNP.PA",
    "SU.PA", "DG.PA", "EL.PA", "RMS.PA", "KER.PA", "SGO.PA",
    "VIE.PA", "CAP.PA", "DSY.PA", "BN.PA", "ENGI.PA", "ORA.PA",
    "STLAP.PA", "VIV.PA"
]

data = yf.download(tickers_larges, start="2019-01-01", end="2024-01-01")["Close"]

print(data.shape)
print(data.isna().sum().sum())

/tmp/ipykernel_721/2610802156.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers_larges, start="2019-01-01", end="2024-01-01")["Close"]
[*********************100%***********************]  20 of 20 completed

(1283, 20)
0


Ici : Schneider Electric, Vinci, EssilorLuxottica, Hermès, Kering, Saint-Gobain, Veolia, Capgemini, Dassault Systèmes, Danone, Engie, Orange, Stellantis, Vivendi viennent s'ajouter aux 6 précédentes — 20 actions au total, secteurs variés. .isna().sum().sum() donne directement le total de valeurs manquantes toutes colonnes confondues.

on refait exactement les mêmes calculs qu'avant, mais sur ce data élargi :

In [ ]:
momentum_6m = data.pct_change(periods=126) * 100
future_return_6m = (data.shift(-126) / data - 1) * 100

dataset = pd.DataFrame({
    "momentum_6m": momentum_6m.stack(),
    "future_return_6m": future_return_6m.stack()
}).dropna()

print(dataset.shape)
print(dataset["momentum_6m"].corr(dataset["future_return_6m"]))

(20620, 2)
-0.052663057857959564


Résultat intéressant: 20 620 observations, mais une corrélation qui s'affaiblit nettement, passant de -0,18 (sur 6 actions) à -0,053 (sur 20 actions) — toujours négative, donc le sens de l'effet reste cohérent, mais son intensité apparente a presque divisé par 3-4.

Même logique qu'avant avec les tertiles, juste avec 5 groupes au lieu de 3. La granularité plus fine permet de mieux voir si l'effet est vraiment progressif (Q1 > Q2 > Q3 > Q4 > Q5) ou s'il n'est porté que par les extrêmes.

In [ ]:
dataset["quintile"] = dataset.groupby("Date")["momentum_6m"].transform(
    lambda x: pd.qcut(x, 5, labels=["Q1 (faible)", "Q2", "Q3", "Q4", "Q5 (fort)"], duplicates="drop")
)

resultats = dataset.groupby("quintile")["future_return_6m"].mean()
print(resultats)

quintile
Q1 (faible)    10.002522
Q2              8.655574
Q3              6.740978
Q4              7.827298
Q5 (fort)       5.223319
Name: future_return_6m, dtype: float64


/tmp/ipykernel_721/3967086505.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resultats = dataset.groupby("quintile")["future_return_6m"].mean()


In [ ]:
dataset = dataset.reset_index()

train = dataset[dataset["Date"] < "2022-07-01"]
test = dataset[dataset["Date"] >= "2022-07-01"]

X_train, y_train = train[["momentum_6m"]], train["future_return_6m"]
X_test, y_test = test[["momentum_6m"]], test["future_return_6m"]

modele_large = LinearRegression()
modele_large.fit(X_train, y_train)

predictions = modele_large.predict(X_test)
mae_large = mean_absolute_error(y_test, predictions)
r2_large = r2_score(y_test, predictions)

mae_baseline_large = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

print("Coefficient :", modele_large.coef_)
print("R² :", r2_large)
print("MAE modèle :", mae_large)
print("MAE baseline :", mae_baseline_large)

Coefficient : [-0.02117841]
R² : -0.028352613648755742
MAE modèle : 10.704170387888244
MAE baseline : 10.813285189663464


C'est le même enchaînement qu'avant : découpage chronologique, entraînement, évaluation sur le test, comparaison à la baseline naïve  mais appliqué au dataset du panier de 20 actions.

calcul de la volatilité :

data.pct_change() calcule le rendement quotidien (d'un jour sur l'autre), sans fenêtre — une nouveauté par rapport à ce qu'on a fait jusqu'ici avec periods=126.
.rolling(window=126) crée une fenêtre glissante de 126 jours qui se déplace le long du temps.
.std() calcule l'écart-type des rendements quotidiens à l'intérieur de chaque fenêtre — une mesure directe de la volatilité.
* (252 ** 0.5) annualise cette volatilité (252 étant le nombre approximatif de jours de bourse par an) — une convention standard en finance pour rendre les volatilités comparables entre elles, quelle que soit la fréquence de calcul.

In [ ]:
rendements_quotidiens = data.pct_change()
volatilite_6m = rendements_quotidiens.rolling(window=126).std() * (252 ** 0.5) * 100

print(volatilite_6m.tail())

Ticker          AI.PA      BN.PA     BNP.PA     CAP.PA      DG.PA     DSY.PA  \
Date                                                                           
2023-12-21  15.147960  13.675174  20.298957  23.693857  14.808752  22.843376   
2023-12-22  15.144554  13.675024  20.177021  23.581021  14.827193  22.844422   
2023-12-27  15.133904  13.565278  20.056733  23.457470  14.798754  22.640188   
2023-12-28  15.144856  13.552984  20.032090  23.482056  14.842989  22.621785   
2023-12-29  15.108823  13.496564  19.984037  23.411741  14.837595  22.612629   

Ticker          EL.PA    ENGI.PA     KER.PA      MC.PA      OR.PA     ORA.PA  \
Date                                                                           
2023-12-21  17.511873  15.133728  26.951005  27.038843  18.153216  12.849931   
2023-12-22  16.965468  14.021046  26.967859  27.036845  18.076621  12.450295   
2023-12-27  16.667432  13.869221  27.004383  26.998384  17.914872  12.493877   
2023-12-28  16.572352  13.945586  27.03

In [ ]:
dataset_vol = pd.DataFrame({
    "volatilite_6m": volatilite_6m.stack(),
    "future_return_6m": future_return_6m.stack()
}).dropna()

correlation_vol = dataset_vol["volatilite_6m"].corr(dataset_vol["future_return_6m"])

dataset_vol["quintile_vol"] = dataset_vol.groupby("Date")["volatilite_6m"].transform(
    lambda x: pd.qcut(x, 5, labels=["Q1 (faible vol)", "Q2", "Q3", "Q4", "Q5 (forte vol)"], duplicates="drop")
)
resultats_vol = dataset_vol.groupby("quintile_vol")["future_return_6m"].mean()

print("Corrélation volatilité / rendement futur :", correlation_vol)
print(resultats_vol)

Corrélation volatilité / rendement futur : 0.06943863512705638
quintile_vol
Q1 (faible vol)    10.375799
Q2                  5.449134
Q3                  6.060535
Q4                  9.137024
Q5 (forte vol)      7.427829
Name: future_return_6m, dtype: float64


/tmp/ipykernel_721/1893400834.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resultats_vol = dataset_vol.groupby("quintile_vol")["future_return_6m"].mean()


Le signe : la corrélation est positive (+0,069) — donc plus une action est volatile, plus son rendement futur tend à être élevé. C'est l'inverse de la "low volatility anomaly" (qui prédit une relation négative : moins de volatilité, meilleur rendement ajusté au risque).

Les quintiles confirment surtout une absence de tendance claire : 10,38 % → 5,45 % → 6,06 % → 9,14 % → 7,43 %. Ce n'est ni une décroissance, ni une croissance régulière — ça ressemble davantage à une variation sans ordre logique qu'à un vrai effet progressif comme on l'avait pour le momentum.